# Penyelesaian SPL 3 Variabel — Eliminasi Gauss (Pivoting Parsial)
Nama: (isi nama) · NIM: (isi NIM)

**Versi dengan penjelasan detail tiap baris / cell.**

Alur notebook:
1. Import library
2. Masukkan data SPL
3. Susun matriks augmented [A | B]
4. Eliminasi maju (dengan pivoting parsial) → matriks segitiga atas
5. Substitusi balik → nilai x, y, z
6. Verifikasi hasil (residu + numpy)
7. Uji dengan SPL lain, termasuk kasus tanpa solusi tunggal

Jalankan cell secara berurutan dari atas ke bawah (Runtime → Run all, atau Shift+Enter per cell).

## Cell 1 — Import library
`numpy` hanya dipakai di akhir untuk **membandingkan** hasil. Perhitungan utama ditulis manual supaya langkah eliminasi Gauss terlihat jelas.

In [ ]:
# Mengimpor library numpy dengan nama singkat "np".
# numpy menyediakan np.linalg.solve yang dipakai sebagai pembanding hasil di Cell 6.
import numpy as np

## Cell 2 — Data SPL
Contoh SPL:

2x + y − z = 8
−3x − y + 2z = −11
−2x + y + 2z = −3

**A** = matriks koefisien (3×3), **B** = vektor konstanta di ruas kanan.

In [ ]:
# Matriks koefisien A: tiap baris = koefisien x, y, z dari satu persamaan.
A = [[ 2,  1, -1],   # persamaan 1:  2x +  y -  z
     [-3, -1,  2],   # persamaan 2: -3x -  y + 2z
     [-2,  1,  2]]   # persamaan 3: -2x +  y + 2z

# Vektor konstanta B: ruas kanan dari persamaan 1, 2, 3 (urutannya harus sama dengan A).
B = [8, -11, -3]

# Menampilkan data supaya bisa dicek sebelum dihitung.
print("Matriks A:", A)
print("Vektor B :", B)

## Cell 3 — Matriks augmented
Matriks augmented menggabungkan A dan B menjadi satu matriks 3×4: tiga kolom pertama koefisien, kolom terakhir konstanta.
Fungsi `tampil` dipakai untuk mencetak matriks dengan rapi di setiap tahap.

In [ ]:
def buat_augmented(A, B):
    # Untuk tiap baris i (0, 1, 2), ambil 3 koefisien A[i] lalu tempelkan B[i] di ujung.
    # float(...) mengubah bilangan bulat menjadi desimal supaya pembagian tidak terpotong.
    return [[float(A[i][0]), float(A[i][1]), float(A[i][2]), float(B[i])] for i in range(3)]

def tampil(M, judul=""):
    # Cetak judul (jika ada) di atas matriks.
    if judul:
        print(judul)
    # Cetak tiap baris: 3 koefisien, garis "|", lalu konstanta.
    # Format {v:9.4f} = lebar 9 karakter, 4 angka di belakang koma, supaya kolom sejajar.
    for baris in M:
        print("  [" + "  ".join(f"{v:9.4f}" for v in baris[:3]) + " | " + f"{baris[3]:9.4f}" + " ]")
    print()   # baris kosong sebagai pemisah

# Membuat matriks augmented dari data Cell 2 lalu menampilkannya.
M = buat_augmented(A, B)
tampil(M, "Matriks augmented awal:")

## Cell 4 — Eliminasi maju dengan pivoting parsial
Tujuan: membuat semua elemen **di bawah diagonal** menjadi 0 (matriks segitiga atas).

Untuk tiap kolom pivot k (kolom 1 lalu kolom 2):
1. **Pivoting parsial**: cari baris dengan nilai mutlak terbesar di kolom k, lalu tukar ke posisi pivot. Ini mencegah pembagian dengan nol dan mengurangi galat pembulatan.
2. **Eliminasi**: kurangi baris-baris di bawahnya dengan (faktor × baris pivot) sehingga elemen kolom k menjadi 0.

In [ ]:
# Toleransi: nilai yang lebih kecil dari ini dianggap nol
# (bilangan desimal di komputer jarang tepat 0 karena galat pembulatan).
EPS = 1e-12

def eliminasi_maju(M):
    # Salin matriks supaya matriks asli M tidak ikut berubah.
    M = [baris[:] for baris in M]

    # k = indeks kolom pivot: 0 (kolom x) lalu 1 (kolom y). Kolom z tidak perlu dieliminasi.
    for k in range(2):
        # PIVOTING: cari indeks baris p (dari k sampai 2) dengan |M[p][k]| paling besar.
        p = max(range(k, 3), key=lambda i: abs(M[i][k]))

        # Jika pivot terbesar pun ≈ 0, seluruh kolom ini nol → determinan 0 → tidak ada solusi tunggal.
        if abs(M[p][k]) < EPS:
            raise ValueError("SPL tidak punya solusi tunggal (determinan = 0)")

        # Tukar baris pivot ke posisi k jika belum di sana (cukup dengan menukar dua elemen list).
        if p != k:
            M[k], M[p] = M[p], M[k]
            print(f"Tukar baris {k+1} <-> {p+1}")

        # ELIMINASI: untuk setiap baris i di bawah baris pivot ...
        for i in range(k + 1, 3):
            # faktor = berapa kali baris pivot harus dikurangkan supaya M[i][k] menjadi 0.
            faktor = M[i][k] / M[k][k]
            # Kurangi seluruh baris i (dari kolom k sampai kolom konstanta) dengan faktor × baris pivot.
            for j in range(k, 4):
                M[i][j] -= faktor * M[k][j]

        # Tampilkan matriks setelah kolom k selesai dieliminasi.
        tampil(M, f"Setelah eliminasi kolom {k+1}:")

    # Elemen diagonal terakhir (koefisien z) juga harus bukan nol, kalau tidak z tidak bisa dihitung.
    if abs(M[2][2]) < EPS:
        raise ValueError("SPL tidak punya solusi tunggal (determinan = 0)")

    # Kembalikan matriks segitiga atas.
    return M

# Jalankan eliminasi pada matriks augmented dari Cell 3; hasilnya disimpan di U.
U = eliminasi_maju(M)

## Cell 5 — Substitusi balik
Baris terakhir U hanya berisi z, jadi z dihitung dulu, lalu y dari baris kedua, lalu x dari baris pertama.

In [ ]:
def substitusi_balik(U):
    # Baris 3:  U[2][2]·z = U[2][3]
    z = U[2][3] / U[2][2]
    # Baris 2:  U[1][1]·y + U[1][2]·z = U[1][3]   → pindahkan suku z ke kanan, bagi dengan U[1][1]
    y = (U[1][3] - U[1][2] * z) / U[1][1]
    # Baris 1:  U[0][0]·x + U[0][1]·y + U[0][2]·z = U[0][3]   → pindahkan suku y dan z ke kanan
    x = (U[0][3] - U[0][1] * y - U[0][2] * z) / U[0][0]
    # Kembalikan ketiga nilai sekaligus (tuple).
    return x, y, z

# Hitung solusi dari matriks segitiga atas U (Cell 4).
x, y, z = substitusi_balik(U)

# Tampilkan dengan 6 angka di belakang koma.
print(f"x = {x:.6f}")
print(f"y = {y:.6f}")
print(f"z = {z:.6f}")

## Cell 6 — Verifikasi
Substitusikan x, y, z ke persamaan awal. **Residu** = ruas kiri − ruas kanan; makin mendekati 0, makin tepat solusinya.
Hasil juga dibandingkan dengan `np.linalg.solve`.

In [ ]:
# Untuk setiap persamaan i, hitung ruas kiri a_i1·x + a_i2·y + a_i3·z memakai solusi yang didapat.
for i in range(3):
    kiri = A[i][0] * x + A[i][1] * y + A[i][2] * z
    # Bandingkan dengan ruas kanan B[i]; residu ditulis dalam notasi ilmiah (contoh 1.2e-15 ≈ 0).
    print(f"Persamaan {i+1}: {kiri:.6f} = {B[i]}  (residu {kiri - B[i]:.2e})")

# Pembanding: penyelesaian SPL bawaan numpy. Hasilnya harus sama dengan x, y, z di atas.
print("\nCek dengan numpy.linalg.solve:", np.linalg.solve(np.array(A, float), np.array(B, float)))

## Cell 7 — Coba SPL lain
`selesaikan_spl` membungkus seluruh langkah, jadi SPL lain cukup diselesaikan dengan satu panggilan.
SPL 3 sengaja dibuat **tanpa solusi tunggal** (baris 2 = 2 × baris 1) untuk menguji penanganan error.

In [ ]:
def selesaikan_spl(A, B):
    # Gabungan langkah: augmented → eliminasi maju → substitusi balik.
    U = eliminasi_maju(buat_augmented(A, B))
    return substitusi_balik(U)

# SPL 2:  x + y + z = 6 ;  2y + 5z = -4 ;  2x + 5y - z = 27   (jawaban: x = 5, y = 3, z = -2)
A2 = [[1, 1, 1],
      [0, 2, 5],
      [2, 5, -1]]
B2 = [6, -4, 27]
print("Solusi SPL 2:", selesaikan_spl(A2, B2))

# SPL 3: baris 2 adalah kelipatan baris 1 → determinan 0 → harus muncul pesan error, bukan crash.
A3 = [[1, 2, 3],
      [2, 4, 6],
      [1, 1, 1]]
B3 = [6, 12, 3]
try:
    print(selesaikan_spl(A3, B3))
except ValueError as e:
    # try/except menangkap error dari eliminasi_maju dan menampilkan pesannya.
    print("SPL 3:", e)